# Thesis Phase 1 — AU extraction benchmark (Colab)

**Self-data pilot only** — speed test on `benchmark_5min.mp4` (your `NA_self_*` session).

---

## Before you open Colab (Mac → Google Drive)

Upload to **`MyDrive/THESIS/`** from `~/thesis-phase1/colab/`:

| File / folder | Required |
|---------------|----------|
| `au_benchmark_colab.ipynb` | Open in Colab (File → Upload notebook) |
| `au_benchmark_colab.py` | Yes — re-upload after every repo update |
| `NA_self_20260516_20260516T103935Z/` with `benchmark_5min.mp4` | Yes |
| `reference_facial_au_header.csv` | Optional (column QA) |

---

## Fresh session — run cells in this order

| Step | Cell(s) | Action |
|------|---------|--------|
| **0** | Runtime menu | **Runtime → Change runtime type** → **T4 GPU** → version **2025.07** (not Latest/2026). Disconnect & reconnect if you change runtime. |
| **1** | §1 GPU check | Run. Must print `CUDA: True` and `Tesla T4`. |
| **2** | §2 Python check | Run. If Python **3.12**, change runtime to 3.10/3.11 and restart from §1. |
| **3** | §2b Install | Run once (~2–5 min). Wait for `Install pass`. |
| **4** | §2c Verify | Run. Must show **`numpy 1.26.x`**, **`py-feat 0.6.1`**, **`Detector OK`**. If fail → **2d** → re-run **2c**. |
| **5** | §3 Drive mount | Run. Approve Drive permission. All checks must be **True**. |
| **6** | §3 Option B | **SKIP** (you use Drive). |
| **7** | §4 Benchmark | 5-min clip via subprocess (~30–90 min). **SKIP** if using §4c instead. |
| **8** | §4b DEBUG | **SKIP** unless §4 fails. |
| **9** | §4c Full video | **20-min `recording.mp4`**, chunked + **tqdm** (~hours). Wait for §4 to finish first. |
| **10** | §5 QA | After benchmark completes. **688 columns**. |
| **11** | §5 Download | Run to save CSV + timing log to your Mac. |

**Skip unless needed:** 2a (pip diagnostic), 2f (numpy 1.26 pin), 2g (0.6.2 patch only).

Settings: **py-feat 0.6.1**, 4 fps, RetinaFace + XGB AU, 688-column output.

## 1. GPU check

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → T4 GPU')

## 2. Install py-feat stack (Colab-native)

Uses **preinstalled Colab torch/CUDA** (do not reinstall torch).

**Common failures:**
- `np.mat was removed in NumPy 2.0` — re-run **2b** (pins numpy 1.26), **Restart runtime**, **2c**.
- Pip **dependency conflicts** after 2b — **expected** for `nltools`/`thinc`/`tsfresh`; ignore if **2c** passes. Critical pins: **numpy 1.26**, **opencv 4.8.1.78**, **numexpr &lt;2.8.5**.
- `metadata-generation-failed` — use Python **3.10/3.11** runtime, not 3.12.
- `ModuleNotFoundError: pynv` — run **2d**, re-run **2c**.

**Diagnostic:** 2a. **Install:** 2b → verify 2c. **Hotfix:** 2d. **Emergency:** 2e (uninstall tables).

In [ ]:
# 2a — OPTIONAL: run only if install fails; last lines show the broken package
# !pip install -v py-feat 2>&1 | grep -E 'Collecting |Downloading |Preparing metadata|error:|subprocess-exited'

In [ ]:
import sys
print('Python:', sys.version)
if sys.version_info >= (3, 12):
    print('→ Use Runtime → Change runtime type → Python 3.10 (recommended), then re-run from §1 GPU check.')

In [ ]:
# 2b — Pin stack for py-feat 0.6.1 (numpy 1.26 + numexpr 2.8.4 + opencv 4.8)
# Do NOT reinstall torch. Pip WILL warn about thinc/tsfresh/albumentations — ignore if 2c passes.

# Remove Colab versions that fight our pins
!pip uninstall -y numexpr opencv-python-headless opencv-python 2>/dev/null || true

# Core numeric stack (py-feat needs np.mat → numpy 1.x; scipy 1.13 keeps simps)
!pip install -q --force-reinstall --only-binary=numpy,scipy "numpy==1.26.4" "scipy==1.13.1"

# Force exact versions pip keeps upgrading (use --no-deps so resolver cannot bump them)
!pip install -q --force-reinstall --no-deps "numexpr==2.8.4"
!pip install -q --force-reinstall --no-deps "opencv-python-headless==4.8.1.78"

!pip install -q --no-deps "py-feat==0.6.1" "nltools==0.5.1"
!pip install -q --no-deps "tables==3.9.2" pynv \
    "pywavelets>=0.3.0" "h5py>=2.7.0" "Pillow>=6.0.0" \
    "scikit-learn>=1.2" "scikit-image>=0.19" "joblib" "seaborn>=0.7.0" "matplotlib>=2.1" \
    "easing-functions" "celluloid" "kornia" "av>=9.2.0" "xgboost>=1.6.0" \
    nibabel nilearn
!pip install -q pandas tqdm

import numpy as np, numexpr, cv2
print('Pinned:', 'numpy', np.__version__, '| numexpr', numexpr.__version__, '| cv2', cv2.__version__)
assert np.__version__.startswith('1.26'), 'numpy pin failed — Restart runtime, re-run 2b'
assert numexpr.__version__.startswith('2.8.'), f'numexpr pin failed ({numexpr.__version__}) — re-run 2b'
print('Install pass — run verify cell 2c next')

In [ ]:
# 2c — Verify (minimal)
import sys
import numpy as np
import scipy
import scipy.integrate as _si
import cv2
import numexpr
import pandas as pd
import pynv
import torch

try:
    import tables
    tables_ver = tables.__version__
except AttributeError as e:
    raise RuntimeError(
        '_ARRAY_API / broken pytables — run hotfix cell 2d (upgrade numexpr+tables) or 2e (uninstall tables)'
    ) from e
except ImportError:
    tables_ver = 'not installed (OK for detect_video)'

import feat
from feat import Detector

assert hasattr(_si, 'simps'), 'scipy too new — re-run install (need scipy==1.13.1, <1.14)'
assert feat.__version__.startswith('0.6.1'), (
    f'Wrong py-feat {feat.__version__} — run: !pip install -q --no-deps py-feat==0.6.1 then Restart runtime'
)
assert int(np.__version__.split('.')[0]) < 2, (
    f'NumPy {np.__version__} breaks py-feat (np.mat removed). Re-run cell 2b, then Runtime → Restart session'
)
# Same model stack as Windows / au_benchmark_colab.py (no identity model on 0.6.1)
Detector(
    face_model='retinaface',
    landmark_model='mobilefacenet',
    au_model='xgb',
    emotion_model='resmasknet',
    facepose_model='img2pose',
)

print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
print('numpy', np.__version__, '| scipy', scipy.__version__)
print('numexpr', numexpr.__version__, '| tables', tables_ver)
print('cv2', cv2.__version__, '| pandas', pd.__version__, '| pynv', getattr(pynv, '__version__', 'ok'))
print('py-feat', feat.__version__, '| Detector OK')

torch.set_grad_enabled(False)

In [ ]:
# 2d — HOTFIX: numexpr still 2.14 or opencv still 4.12 after 2b
!pip uninstall -y numexpr opencv-python-headless 2>/dev/null || true
!pip install -q --force-reinstall --no-deps "numexpr==2.8.4" "opencv-python-headless==4.8.1.78"
!pip install -q pynv
import numexpr, cv2
print('numexpr', numexpr.__version__, '| cv2', cv2.__version__, '— re-run 2c')

In [ ]:
# 2e — EMERGENCY: uninstall pytables if 2d still raises _ARRAY_API
# nltools only uses tables for legacy Brain_Data HDF loads; detect_video does not need it.
# attempt_to_import catches ImportError only — a broken tables install must be upgraded (2d) or removed.

!pip uninstall -y tables 2>/dev/null || true
print('tables uninstalled — re-run verify cell 2c, then benchmark')

In [ ]:
# 2f — SKIP (numpy 1.26 pin is now in cell 2b). Only run if you skipped 2b.
# !pip install -q --force-reinstall --only-binary=numpy,scipy "numpy==1.26.4" "scipy==1.13.1"
# !pip install -q "numexpr>=2.7.3,<2.8.5" opencv-python-headless==4.8.1.78
print('Use cell 2b instead of 2f')

In [ ]:
# 2g — OPTIONAL: only if you kept py-feat==0.6.2 (KeyError: 'representation_model')
import feat
if getattr(feat, '__version__', '').startswith('0.6.2'):
    import importlib
    from pathlib import Path
    import inspect
    import feat.pretrained as pt
    pre = Path(inspect.getfile(pt))
    t = pre.read_text()
    broken = 'if identity_model is None:\n        raise ValueError(\n            f"representation_model must be one of'
    if broken in t:
        t = t.replace(broken, 'if identity_model is None:\n        pass\n    elif False:\n        raise ValueError(\n            f"representation_model must be one of', 1)
        pre.write_text(t)
        importlib.reload(pt)
    # Full init/detect_video patches: re-upload au_benchmark_colab.py and run:
    # from au_benchmark_colab import _patch_pyfeat_identity_none; _patch_pyfeat_identity_none()
    print('pretrained.py patched — re-upload au_benchmark_colab.py and use load_detector() for full fix')
else:
    print('py-feat', feat.__version__, '— no 0.6.2 patch needed (use 0.6.1 per cell 2b)')

## 3. Drive paths & mount

**Your layout (Google Drive):**

| Path | Contents |
|------|----------|
| `MyDrive/THESIS/NA_self_20260516_20260516T103935Z/` | Session data + `benchmark_5min.mp4` |
| `MyDrive/THESIS/au_benchmark_colab.py` | Benchmark script (re-upload when repo updates) |
| `MyDrive/THESIS/reference_facial_au_header.csv` | Optional 688-col header QA |

Prepare on Mac (once):
```bash
cd ~/thesis-phase1/sessions/NA_self_20260516_20260516T103935Z
ffmpeg -y -ss 60 -i recording.mp4 -t 300 -c copy benchmark_5min.mp4
```

In [ ]:
from pathlib import Path
from google.colab import drive

# --- Single source of truth: your Google Drive THESIS folder ---
drive.mount('/content/drive')

SESSION_DIR = '/content/drive/MyDrive/THESIS/NA_self_20260516_20260516T103935Z'
BENCHMARK_SCRIPT = Path('/content/drive/MyDrive/THESIS/au_benchmark_colab.py')
REFERENCE_HEADER = Path('/content/drive/MyDrive/THESIS/reference_facial_au_header.csv')

VIDEO_FILE = 'benchmark_5min.mp4'  # or 'recording.mp4' for full 20-min (slow)
FPS = 4
BATCH_SIZE = 4  # use 1 if CUDA OOM
USE_CHUNKS = False  # False for 5-min clip; True for full recording.mp4

p = Path(SESSION_DIR)
print('Session exists:', p.is_dir(), '→', SESSION_DIR)
print('Video exists:', (p / VIDEO_FILE).is_file())
print('Script exists:', BENCHMARK_SCRIPT.is_file(), '→', BENCHMARK_SCRIPT)
print('Reference header:', REFERENCE_HEADER.is_file())
assert p.is_dir(), f'Missing session folder on Drive: {SESSION_DIR}'
assert (p / VIDEO_FILE).is_file(), f'Missing {VIDEO_FILE} in session folder'
assert BENCHMARK_SCRIPT.is_file(), f'Upload au_benchmark_colab.py to MyDrive/THESIS/'

In [ ]:
# SKIP if using Google Drive (§3 above). Only for zip upload to /content/.
# --- Option B: direct upload (small 5-min clip) ---
if not Path(SESSION_DIR).is_dir():
    from google.colab import files
    import zipfile, io
    print('Upload a zip of the session folder (must contain benchmark_5min.mp4)')
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.endswith('.zip'):
            z = zipfile.ZipFile(io.BytesIO(data))
            z.extractall('/content')
            print('Extracted to /content/')
    print('Set SESSION_DIR above to the extracted folder path, then re-run this cell.')

## 4. Run benchmark (~30–90 min)

Runs `au_benchmark_colab.py` on Drive. Temp files go to `/content/colab_work/` (not Drive). Final CSV: `facial_au_colab.csv` in your session folder on Drive.

**While running:** you should see `GPU:`, `py-feat 0.6.1`, `Work dir (local temp):`, then JSON timing lines. Do not stop the cell unless you intend to cancel.

In [ ]:
import subprocess
import sys

script = BENCHMARK_SCRIPT  # /content/drive/MyDrive/THESIS/au_benchmark_colab.py

cmd = [
    sys.executable, str(script),
    '--session-dir', SESSION_DIR,
    '--input', VIDEO_FILE,
    '--fps', str(FPS),
    '--batch-size', str(BATCH_SIZE),
]
if REFERENCE_HEADER.is_file():
    cmd.extend(['--reference-csv', str(REFERENCE_HEADER)])
if not USE_CHUNKS:
    cmd.append('--no-chunks')

print(' '.join(cmd))
print('--- subprocess output ---')
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f'Benchmark exited with code {result.returncode}. '
        'If stdout/stderr are empty, run the DEBUG cell below (§4b).'
    )

### 4b. DEBUG — in-process run (shows full traceback)

Run this if §4 prints `returncode=1` with empty output. Re-upload latest `au_benchmark_colab.py` first.

In [ ]:
import sys
import runpy

argv = [
    str(BENCHMARK_SCRIPT),
    '--session-dir', SESSION_DIR,
    '--input', VIDEO_FILE,
    '--fps', str(FPS),
    '--batch-size', str(BATCH_SIZE),
]
if REFERENCE_HEADER.is_file():
    argv.extend(['--reference-csv', str(REFERENCE_HEADER)])
if not USE_CHUNKS:
    argv.append('--no-chunks')

sys.argv = argv
print(' '.join(argv))
runpy.run_path(str(BENCHMARK_SCRIPT), run_name='__main__')

## 4c. Full 20-min `recording.mp4` (chunked, with progress)

**Run in-process** so you see:
- **Session chunks** bar (4 × 5 min for a 20 min video)
- **py-feat frame** bar inside each chunk (`283` frames ≈ 5 min @ 4 fps)

Requires §2c verify passed. Uses `recording.mp4` on Drive. Temp work: `/content/colab_work/...` (resumable `chunk_XXX.csv`).

**Do not run while §4 is still executing** — wait or Interrupt §4 first.

Output: `facial_au_full_colab.csv` on Drive (~4800 rows). Expect **several hours** on T4.

In [ ]:
import importlib.util
import sys
from pathlib import Path

# Load benchmark module from Drive (re-upload .py after repo updates)
assert BENCHMARK_SCRIPT.is_file(), f'Missing {BENCHMARK_SCRIPT}'
spec = importlib.util.spec_from_file_location('au_benchmark_colab', BENCHMARK_SCRIPT)
bench = importlib.util.module_from_spec(spec)
sys.modules['au_benchmark_colab'] = bench
spec.loader.exec_module(bench)

FULL_VIDEO = 'recording.mp4'
FULL_OUTPUT = 'facial_au_full_colab.csv'
CHUNK_MINUTES = 5  # 4 chunks for ~20 min

assert (Path(SESSION_DIR) / FULL_VIDEO).is_file(), f'Need {FULL_VIDEO} in session folder'

print('Starting chunked full-video run (tqdm should appear below)...')
bench.run_benchmark(
    Path(SESSION_DIR),
    video_name=FULL_VIDEO,
    fps=FPS,
    batch_size=BATCH_SIZE,
    chunk_seconds=CHUNK_MINUTES * 60,
    use_chunks=True,
    output_name=FULL_OUTPUT,
    reference_csv=REFERENCE_HEADER if REFERENCE_HEADER.is_file() else None,
    show_progress=True,
    resume_chunks=True,
)
print('Done:', Path(SESSION_DIR) / FULL_OUTPUT)

## 5. Timing log & QA (688 columns)

In [ ]:
import json
import pandas as pd
from pathlib import Path

session = Path(SESSION_DIR)
# Timing log lives in local work dir when session is on Drive (see au_benchmark_colab.py)
log_path = Path('/content/colab_work') / session.name / 'benchmark_timing.jsonl'
if not log_path.is_file():
    log_path = session / 'colab_work' / 'benchmark_timing.jsonl'
out_csv = session / 'facial_au_colab.csv'

print('=== Timing log ===')
if log_path.is_file():
    for line in log_path.read_text().strip().splitlines():
        print(json.dumps(json.loads(line), indent=2))
else:
    print('No log yet:', log_path)

print('\n=== Output QA ===')
if not out_csv.is_file():
    raise FileNotFoundError(f'No output yet: {out_csv} — complete §4 first')
df = pd.read_csv(out_csv)
print('Rows:', len(df), '| Columns:', len(df.columns), '(expect 688)')
print('timestamp_utc present:', 'timestamp_utc' in df.columns)
if len(df.columns) != 688:
    print('WARNING: column count mismatch vs Windows baseline')

# Optional: reference header on Drive (REFERENCE_HEADER from §3) or /content/
ref = REFERENCE_HEADER if REFERENCE_HEADER.is_file() else Path('/content/reference_facial_au_header.csv')
if ref.is_file():
    ref_cols = ref.read_text().splitlines()[0].split(',')
    print('Header match:', list(df.columns) == ref_cols)

In [ ]:
# Download results
from google.colab import files
files.download(str(out_csv))
if log_path.is_file():
    files.download(str(log_path))